# [SK 07.5 - AI Foundry Agents with Bing using Declarative Spec](https://learn.microsoft.com/en-us/semantic-kernel/frameworks/agent/agent-types/azure-ai-agent?pivots=programming-language-python#declarative-spec)
**Note**: `azure-ai-agents==1.1.0b4` and `azure-ai-projects==1.0.0` are automatically installed by semantic kernel 1.35.0.<br/>

The AzureAIAgent supports instantiation from a YAML declarative specification. The declarative approach allows you to define the agent's properties, instructions, model configuration, tools, and other options in a single, auditable document. This makes agent composition portable and easily managed across environments.<br/>
A minimal YAML declarative spec might look like the following:
```
type: foundry_agent
name: sk_aifoundry_agent-PYTHON-from-specs
instructions: You are a clever agent
description: This agent answers questions
model:
  id: gpt-4o
  options:
    temperature: 0.4
tools:
  - id: LightsPlugin.get_lights
    type: function
  - id: LightsPlugin.change_state
    type: function
```

# Constants and Libraries

Just if needed, log on Azure
```
import os

# Login with tenant ID
os.system("az login --tenant 3ad0b905-34ab-4116-93d9-c1dcc2d35af6 --output none") # --use-device-code

# Set the subscription programmatically
os.system("az account set --subscription eca2eddb-0f0c-4351-a634-52751499eeea")
```

In [1]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
import importlib.metadata

if not load_dotenv("./../config/credentials_my.env"):
    print("Environment variables not loaded, cell execution stopped")
else:
    print("Environment variables have been loaded ;-)")

agent_name = "sk_aifoundry_agent-PYTHON-from-specs"

instructions  = "You are a clever agent"
description   = "This agent answers questions" #  using Bing to provide grounding context.

project_endpoint = os.environ["AIF_BAS_PROJECT_ENDPOINT"] # AIF_BAS_PROJECT_ENDPOINT or AIF_STD_PROJECT_ENDPOINT
deployment_name =  os.environ["MODEL_DEPLOYMENT_NAME"]
openai_api_version = os.environ["OPENAI_API_VERSION"] # not less than 2025-03-01-preview
openai_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]

credential = DefaultAzureCredential()

print(f'OpenAI Endpoint: {openai_endpoint}')
print(f'Project Endpoint: {project_endpoint}')
print(f'OpenAI API Version: {openai_api_version}')
print(f"azure-ai-projects library installed version: {importlib.metadata.version("azure-ai-projects")}")
print(f"azure-ai-agents library installed version: {importlib.metadata.version("azure-ai-agents")}")

Environment variables have been loaded ;-)
OpenAI Endpoint: https://mmoaiswc-01.openai.azure.com/
Project Endpoint: https://aif1bassvj36b.services.ai.azure.com/api/projects/aif1basswcprj01
OpenAI API Version: 2025-04-01-preview
azure-ai-projects library installed version: 1.0.0
azure-ai-agents library installed version: 1.1.0b4


# Create AI FOUNDRY PROJECT CLIENT using Semantic Kernel SDK

In [2]:
from semantic_kernel.agents import AzureAIAgent, AzureAIAgentSettings
os.environ["AZURE_AI_AGENT_ENDPOINT"] = project_endpoint
os.environ["AZURE_AI_AGENT_MODEL_DEPLOYMENT_NAME"] =  deployment_name

credential = DefaultAzureCredential()

project_client = AzureAIAgent.create_client(credential=DefaultAzureCredential())
agent_settings = AzureAIAgentSettings() # other than "from semantic_kernel.connectors.ai.open_ai import AzureOpenAISettings"

# Native Plugin

In [3]:
class LightsPlugin:
    from typing import Annotated
    from semantic_kernel.functions import kernel_function
   
    def __init__(self):
        self.lights = [
        {"id": 0, "name": "Table Lamp", "is_on": False},
        {"id": 1, "name": "Porch light", "is_on": False},
        {"id": 2, "name": "Chandelier", "is_on": False},]
 
    @kernel_function(
        name="get_lights", # <<<=== DIFFERENT FROM THE FUNCTION NAME <get_state>, which will be ignored
        description="Gets a list of lights and their current state",
    )
    def get_state(
        self,
    ) -> Annotated[str, "the output is a string"]:
        """Gets a list of lights and their current state."""
        return self.lights
 
    @kernel_function(
        name="change_state",
        description="Changes the state of the light",
    )
    def change_state(
        self,
        id: int,
        is_on: bool,
    ) -> Annotated[str, "the output is a string"]:
        """Changes the state of the light."""
        for light in self.lights:
            if light["id"] == id:
                light["is_on"] = is_on
                return light
        return None

# Create an AI Foundry Agent

## Define the YAML specification string

In [4]:
spec = f"""
type: foundry_agent
name: {agent_name}
instructions: {instructions}
description: {description}
model:
  id: {agent_settings.model_deployment_name}
  options:
    temperature: 0.4
tools:
  - id: LightsPlugin.get_lights
    type: function
  - id: LightsPlugin.change_state
    type: function
"""

print(spec)


type: foundry_agent
name: sk_aifoundry_agent-PYTHON-from-specs
instructions: You are a clever agent
description: This agent answers questions
model:
  id: gpt-4o
  options:
    temperature: 0.4
tools:
  - id: LightsPlugin.get_lights
    type: function
  - id: LightsPlugin.change_state
    type: function



## Create the AzureAI Agent from the YAML spec

In [5]:
from semantic_kernel.agents import AgentRegistry

agent: AzureAIAgent = await AgentRegistry.create_from_yaml(
    yaml_str=spec,
    client=project_client,
    plugins=[LightsPlugin()],
    settings=AzureAIAgentSettings(),
)

# Interacting with an AzureAIAgent
Interaction with the AzureAIAgent is straightforward. The agent maintains the conversation history automatically using a thread.<br/>
The specifics of the Azure AI Agent thread is abstracted away via the AzureAIAgentThread class, which is an implementation of AgentThread.

In [6]:
from semantic_kernel.agents import AzureAIAgentThread
from semantic_kernel.contents import AuthorRole

USER_INPUTS = [
    "Hello", 
    "Please toggle the porch light", 
    "What's the status of all lights?", 
    "Thank you",
]

thread: AzureAIAgentThread = None

try:
    i=0
    for user_input in USER_INPUTS:
        i+=1
        print(f"Message {i} from {AuthorRole.USER}: '{user_input}'")
        response = await agent.get_response(messages=user_input, thread=thread)
        print(f"Message {i} from {AuthorRole.ASSISTANT}): '{response}'\n")
        thread = response.thread
finally:
    if thread:
        print(f"\nThread <{thread.id}> was created to manage the conversation")

Message 1 from AuthorRole.USER: 'Hello'
Message 1 from AuthorRole.ASSISTANT): 'Hello! How can I assist you today?'

Message 2 from AuthorRole.USER: 'Please toggle the porch light'
Message 2 from AuthorRole.ASSISTANT): 'The porch light has been turned on. Let me know if there's anything else I can do for you!'

Message 3 from AuthorRole.USER: 'What's the status of all lights?'
Message 3 from AuthorRole.ASSISTANT): 'Here’s the current status of all the lights:

- **Table Lamp**: Off  
- **Porch Light**: On  
- **Chandelier**: Off  

Let me know if you’d like to adjust any!'

Message 4 from AuthorRole.USER: 'Thank you'
Message 4 from AuthorRole.ASSISTANT): 'You're welcome! Feel free to reach out if you need anything else. Have a great day! 😊'


Thread <thread_HitNwffemDz7zfTb5j1NbqnL> was created to manage the conversation


# Teardown

In [7]:
# list and delete all files
file_list = await project_client.agents.files.list()
for file in file_list.data:
    await project_client.agents.files.delete(file_id=file.id)

# project_client.agents.vector_stores.delete(vector_store_id=vector_store.id)

# delete thread
await thread.delete()

# delete agent
await project_client.agents.delete_agent(agent_id=agent.id)